In [0]:
# ============================================================
# CELL 1 — Install Kaggle library
# ============================================================
%pip install kaggle

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================================
# CELL 2 — Install kagglehub and download in one cell
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'kagglehub'], capture_output=True)

import importlib
import kagglehub
importlib.reload(kagglehub)

import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_1c2d14fb213382a4d12f703991b7a01f'

path = kagglehub.competition_download('m5-forecasting-accuracy')

print("Downloaded to:", path)
print("\nFiles:")
for f in os.listdir(path):
    size_mb = os.path.getsize(f'{path}/{f}') / (1024*1024)
    print(f"  {f}  —  {size_mb:.1f} MB")

100%|██████████| 45.8M/45.8M [00:01<00:00, 44.6MB/s]

Extracting files...


Downloaded to: /home/spark-84ff6486-b6e9-4dfc-a8aa-da/.cache/kagglehub/competitions/m5-forecasting-accuracy

Files:
  sales_train_validation.csv  —  114.4 MB
  sales_train_evaluation.csv  —  116.1 MB
  sample_submission.csv  —  5.0 MB
  sell_prices.csv  —  194.0 MB
  calendar.csv  —  0.1 MB


In [0]:
# ============================================================
# CELL 3 — Copy files to Workspace, then read with Spark
# ============================================================
import shutil
import os

cache_path = '/home/spark-84ff6486-b6e9-4dfc-a8aa-da/.cache/kagglehub/competitions/m5-forecasting-accuracy'
workspace_path = '/Workspace/Users/navinnagisetty@gmail.com/walmart-m5'

# Create workspace directory
os.makedirs(workspace_path, exist_ok=True)

# Copy files using Python (not Spark)
files = ['calendar.csv', 'sales_train_validation.csv',
         'sales_train_evaluation.csv', 'sell_prices.csv']

for f in files:
    src = f'{cache_path}/{f}'
    dst = f'{workspace_path}/{f}'
    print(f'Copying {f}...')
    shutil.copy2(src, dst)
    print(f'  Done — {os.path.getsize(dst)/(1024*1024):.1f} MB')

print('\nNow reading into Spark...')

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

df_sales    = spark.read.csv(f'file://{workspace_path}/sales_train_validation.csv', header=True, inferSchema=True)
df_calendar = spark.read.csv(f'file://{workspace_path}/calendar.csv', header=True, inferSchema=True)
df_prices   = spark.read.csv(f'file://{workspace_path}/sell_prices.csv', header=True, inferSchema=True)
df_eval     = spark.read.csv(f'file://{workspace_path}/sales_train_evaluation.csv', header=True, inferSchema=True)

print(f'sales_train_validation : {df_sales.count():,} rows')
print(f'calendar               : {df_calendar.count():,} rows')
print(f'sell_prices            : {df_prices.count():,} rows')
print(f'sales_train_evaluation : {df_eval.count():,} rows')

Files in cache:
  sales_train_validation.csv  — 114.4 MB
  sales_train_evaluation.csv  — 116.1 MB
  sample_submission.csv  — 5.0 MB
  sell_prices.csv  — 194.0 MB
  calendar.csv  — 0.1 MB

Reading with pandas, then converting to Spark...
sales_train_validation : 30,490 rows
calendar               : 1,969 rows
sell_prices            : 6,841,121 rows
sales_train_evaluation : 30,490 rows


In [0]:
# ============================================================
# CELL 4 — Unpivot sales from wide to long format (Bronze)
# ============================================================
from pyspark.sql.functions import col, expr

print("Unpivoting sales data from wide to long format...")
print(f"Input: {df_sales.count():,} rows x {len(df_sales.columns)} columns")

# Get day columns only — everything after the 6 ID columns
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in df_sales.columns if c.startswith('d_')]

print(f"Found {len(day_cols)} day columns to unpivot...")

# Build the unpivot expression
unpivot_expr = f"stack({len(day_cols)}, " + \
               ", ".join([f"'{d}', `{d}`" for d in day_cols]) + \
               ") as (d, sales)"

df_sales_long = df_sales.select(id_cols + [expr(unpivot_expr)])

row_count = df_sales_long.count()
print(f"\nOutput: {row_count:,} rows")
print(f"Columns: {df_sales_long.columns}")
print("\nSample rows:")
df_sales_long.show(5)

Unpivoting sales data from wide to long format...
Input: 30,490 rows x 1919 columns
Found 1913 day columns to unpivot...

Output: 58,327,370 rows
Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales']

Sample rows:
+--------------------+-------------+---------+-------+--------+--------+---+-----+
|                  id|      item_id|  dept_id| cat_id|store_id|state_id|  d|sales|
+--------------------+-------------+---------+-------+--------+--------+---+-----+
|HOBBIES_1_001_CA_...|HOBBIES_1_001|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
|HOBBIES_1_002_CA_...|HOBBIES_1_002|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
|HOBBIES_1_003_CA_...|HOBBIES_1_003|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
|HOBBIES_1_004_CA_...|HOBBIES_1_004|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
|HOBBIES_1_005_CA_...|HOBBIES_1_005|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|    0|
+--------------------+-------------+---------+-------+--------+--------+---+-----+
only sh

In [0]:
# ============================================================
# CELL 5 — Write Bronze Delta tables to Unity Catalog
# ============================================================

print("Writing Bronze Delta tables...")

# Create database/schema to hold all our tables
spark.sql("CREATE SCHEMA IF NOT EXISTS walmart_bronze")

# 1. Sales long format — your main 58M row table
print("Writing bronze_sales (58M rows)...")
df_sales_long.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("walmart_bronze.bronze_sales")
print("  Done")

# 2. Calendar
print("Writing bronze_calendar...")
df_calendar.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("walmart_bronze.bronze_calendar")
print("  Done")

# 3. Sell prices
print("Writing bronze_prices (6.8M rows)...")
df_prices.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("walmart_bronze.bronze_prices")
print("  Done")

# 4. Evaluation sales
print("Writing bronze_sales_eval...")
df_eval.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("walmart_bronze.bronze_sales_eval")
print("  Done")

print("\nAll Bronze Delta tables written successfully.")
print("\nVerifying tables exist:")
spark.sql("SHOW TABLES IN walmart_bronze").show()

Writing Bronze Delta tables...
Writing bronze_sales (58M rows)...
  Done
Writing bronze_calendar...
  Done
Writing bronze_prices (6.8M rows)...
  Done
Writing bronze_sales_eval...
  Done

All Bronze Delta tables written successfully.

Verifying tables exist:
+--------------+-----------------+-----------+
|      database|        tableName|isTemporary|
+--------------+-----------------+-----------+
|walmart_bronze|  bronze_calendar|      false|
|walmart_bronze|    bronze_prices|      false|
|walmart_bronze|     bronze_sales|      false|
|walmart_bronze|bronze_sales_eval|      false|
+--------------+-----------------+-----------+



In [0]:
# ============================================================
# CELL 6 — Bronze validation checks
# ============================================================

print("=" * 50)
print("BRONZE LAYER VALIDATION")
print("=" * 50)

# Row counts
print("\nRow counts:")
for table in ['bronze_sales', 'bronze_calendar', 
              'bronze_prices', 'bronze_sales_eval']:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM walmart_bronze.{table}") \
                 .collect()[0]['cnt']
    print(f"  {table:<25} {count:>12,} rows")

# Check date range
print("\nSales date range:")
spark.sql("""
    SELECT 
        MIN(d) as first_day,
        MAX(d) as last_day,
        COUNT(DISTINCT d) as total_days
    FROM walmart_bronze.bronze_sales
""").show()

# Check stores and states
print("Stores and states:")
spark.sql("""
    SELECT state_id, COUNT(DISTINCT store_id) as stores
    FROM walmart_bronze.bronze_sales
    GROUP BY state_id
    ORDER BY state_id
""").show()

# Check for nulls in sales
print("Null sales values:")
spark.sql("""
    SELECT COUNT(*) as null_sales
    FROM walmart_bronze.bronze_sales
    WHERE sales IS NULL
""").show()

BRONZE LAYER VALIDATION

Row counts:
  bronze_sales                58,327,370 rows
  bronze_calendar                  1,969 rows
  bronze_prices                6,841,121 rows
  bronze_sales_eval               30,490 rows

Sales date range:
+---------+--------+----------+
|first_day|last_day|total_days|
+---------+--------+----------+
|      d_1|   d_999|      1913|
+---------+--------+----------+

Stores and states:
+--------+------+
|state_id|stores|
+--------+------+
|      CA|     4|
|      TX|     3|
|      WI|     3|
+--------+------+

Null sales values:
+----------+
|null_sales|
+----------+
|         0|
+----------+

